# Pipeline integrado do hackathon

Este notebook liga as partes do time:

| Parte (branch) | Pasta | Papel aqui |
|---|---|---|
| `chore/add_dataset_files` | `imagens/` | imagens originais |
| `feat/add_imagens_preprocessadas` | `imagens_preprocessadas/` | binarização do time (entrada `binarizada`) |
| `feat/OCR` | `test_kraken/` | motor Kraken (segmentação neural + PP-OCRv6 medium) |
| `llm_implementation` | `LLM_Transcription.ipynb` | motor LLM (OpenRouter) e métricas de avaliação |
| — | `ground_truth/` | gabarito por página, extraído do DOCX |

Cada combinação (`kraken_binarizada`, `llm_original`, `llm_corrige_kraken_binarizada`...) grava `resultados/<nome>.json`, no mesmo formato do notebook do LLM, e todas são avaliadas pelas mesmas métricas. Detalhes no `README.md`.

**Custos:** Kraken é gratuito. As combinações de LLM ficam bloqueadas enquanto `PERMITIR_GASTO = False`; com `SIMULAR = True` as requisições são montadas, mas não enviadas.

Use um ambiente com GPU (**Ambiente de execução → Alterar o tipo de ambiente de execução → GPU**). Em CPU, o Kraken leva cerca de 1 minuto por página e combinação.

In [ ]:
import os
import sys
from pathlib import Path

BRANCH = 'feat/kraken-finetunado'  # troque para 'main' depois do merge
REPO = Path('/content/hackathon_ocr_historico')
if Path('/content').exists():
    if not REPO.exists():
        !git clone -q -b {BRANCH} https://github.com/fabiolobato/hackathon_ocr_historico.git {REPO}
    else:  # clone de uma execução anterior (talvez de outro branch): atualiza para BRANCH
        !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} pull -q --ff-only origin {BRANCH}
    os.chdir(REPO)
if not Path('ocr_historico/__init__.py').is_file():
    raise RuntimeError(f'ocr_historico/ não encontrado em {Path.cwd()}. Confira BRANCH ({BRANCH}).')
# O Colab não inclui a pasta atual no sys.path depois do chdir
for pasta in (Path.cwd(), Path.cwd() / 'test_kraken'):
    if str(pasta) not in sys.path:
        sys.path.insert(0, str(pasta))
# instala o repositório no ambiente: os imports sobrevivem a "Reiniciar sessão"
%pip -q install -e . --no-deps
!git log -1 --format='%h %s'

## 1. Dependências

Instala as versões fixadas do `test_kraken` e o `jiwer` das métricas. Se aparecer o aviso de versões antigas carregadas, use **Ambiente de execução → Reiniciar sessão** e continue **a partir desta célula**. O repositório já fica instalado no ambiente pela célula anterior, então os imports funcionam depois do reinício.

In [ ]:
import os
from pathlib import Path
from ocr_historico.dados import RAIZ
from kraken_pipeline.setup import configure_notebook, ensure_dependencies

os.chdir(RAIZ)  # depois de reiniciar a sessão, o Colab volta para /content
configure_notebook()
ensure_dependencies(RAIZ / 'test_kraken' / 'requirements.txt')
%pip -q install jiwer==4.0.0

## 2. Páginas, entradas e motor Kraken

As páginas são ligadas pelo número no nome: imagem original, binarização do time e gabarito. `PAGINAS = None` usa as 12; para um teste rápido, use por exemplo `[11]`.

In [ ]:
import logging
from ocr_historico.dados import listar_paginas
from ocr_historico.combinacoes import interpretar, PADRAO_KRAKEN, PADRAO_LLM
from ocr_historico.executar import executar, avaliar_pasta, imprimir_tabela, PASTA_RESULTADOS
from ocr_historico.motor_kraken import MotorKraken, baixar_modelo
from kraken_pipeline.setup import default_device

logging.getLogger('kraken').setLevel(logging.WARNING)
PAGINAS = None
paginas = listar_paginas(PAGINAS)
print([(p.numero, p.binarizada is not None, p.gabarito is not None) for p in paginas])

# Se o Zenodo estiver fora do ar, coloque o medium.safetensors no seu Drive e informe o caminho
# (ex.: '/content/drive/MyDrive/medium.safetensors'). O MD5 é conferido do mesmo jeito.
MODELO_NO_DRIVE = None
if MODELO_NO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
modelo_base = baixar_modelo('medium', copia_local=Path(MODELO_NO_DRIVE) if MODELO_NO_DRIVE else None)
motor_kraken = MotorKraken(modelo_base, default_device())

In [ ]:
from IPython.display import display
from ocr_historico.entradas import carregar_entrada

pagina = paginas[min(6, len(paginas) - 1)]
for entrada in ('original', 'binarizada', 'nlbin'):
    print(entrada)
    display(carregar_entrada(pagina, entrada).resize((330, 450)))

## 3. Kraken com cada entrada

Combinações padrão: imagem original, nlbin do Kraken, binarização do time e as versões ampliadas 2×. Os resultados já transcritos são reaproveitados; use `refazer=True` para gerar de novo.

In [ ]:
COMBINACOES_KRAKEN = list(PADRAO_KRAKEN)

for nome in COMBINACOES_KRAKEN:
    print(executar(interpretar(nome), paginas, PASTA_RESULTADOS, motor_kraken=motor_kraken))

## 3b. Kraken fine-tunado

Usa o modelo treinado no `finetuning_kraken.ipynb` (combinação `kraken_ft_original_2x`). Ele viu as páginas 5, 6, 8, 9, 11 e 13–16 no treino. Na tabela das 12 páginas, o resultado dele aparece marcado como inflado. A comparação justa é a das páginas reservadas (7, 10 e 12), na seção 5.

`MODELO_FT = None` pega o treino mais recente em `finetuning/modelo/` desta sessão. Se o modelo estiver no Drive, informe o caminho.

In [ ]:
from ocr_historico.combinacoes import PADRAO_FINETUNADO
from ocr_historico.finetuning import carregar_finetunado

# ex.: '/content/drive/MyDrive/hackathon_ocr_historico/finetuning/best_0.8182.safetensors'
MODELO_FT = None
if MODELO_FT and MODELO_FT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
try:
    motor_kraken_ft = carregar_finetunado(default_device(), Path(MODELO_FT) if MODELO_FT else None)
    print('Modelo fine-tunado:', motor_kraken_ft.modelo, '| páginas de treino:', motor_kraken_ft.paginas_treino)
    for nome in PADRAO_FINETUNADO:
        print(executar(interpretar(nome), paginas, PASTA_RESULTADOS, motor_kraken_ft=motor_kraken_ft))
except FileNotFoundError as erro:
    motor_kraken_ft = None
    print('Sem modelo fine-tunado; seguindo só com o modelo base.', erro)

## 4. LLM: imagem → LLM e Kraken → LLM

- `llm_original`: o que o notebook `LLM_Transcription.ipynb` já fazia.
- `llm_binarizada`: binarização do time → LLM.
- `llm_corrige_kraken_ft_original_2x`: o LLM recebe a imagem original **e** o texto do Kraken fine-tunado, e corrige. Sem modelo fine-tunado, usa `llm_corrige_kraken_original_2x`.

Chave: segredo `API_KEY` do Colab (como no notebook do LLM), variável `OPENROUTER_API_KEY` ou arquivo `.env`. Custo de referência: ~US$ 0,06 por página e combinação.

Se vocês já têm o JSON gerado pelo `LLM_Transcription.ipynb`, importem-no em vez de pagar de novo:

```python
from ocr_historico.resultados import importar_resultado_llm
importar_resultado_llm(Path('resultados_transcricao (2).json'), PASTA_RESULTADOS, 'llm_original')
```

In [ ]:
from ocr_historico.combinacoes import CORRECAO_SEM_FINETUNING
from ocr_historico.motor_llm import MotorLLM

PERMITIR_GASTO = False  # só mude depois de aprovar o custo
SIMULAR = not PERMITIR_GASTO
COMBINACOES_LLM = [c if globals().get('motor_kraken_ft') or 'kraken_ft' not in c else CORRECAO_SEM_FINETUNING for c in PADRAO_LLM]

motor_llm = MotorLLM(permitir_gasto=PERMITIR_GASTO, simular=SIMULAR)
pasta_llm = PASTA_RESULTADOS / 'simulacao' if SIMULAR else PASTA_RESULTADOS
for nome in COMBINACOES_LLM:
    print(executar(interpretar(nome), paginas, pasta_llm, motor_llm=motor_llm))
print(f'{len(motor_llm.requisicoes_simuladas)} requisições simuladas (nenhuma enviada)' if SIMULAR else 'enviado')

## 5. Comparação

Mesmas páginas e métricas para todas as combinações em `resultados/`. Valores de corpus somam erros de todas as páginas. `wer_estrito` preserva caixa e pontuação (métrica do `test_kraken`). As demais colunas vêm do notebook do LLM. O gabarito ainda é provisório: veja `ground_truth/comparison_review.md`.

In [ ]:
from ocr_historico.finetuning import dividir_paginas

print('TODAS AS PÁGINAS SELECIONADAS')
resumo = avaliar_pasta(PASTA_RESULTADOS, paginas)
imprimir_tabela(resumo)

_, reservadas = dividir_paginas(listar_paginas())
print(f'\nPÁGINAS RESERVADAS {[p.numero for p in reservadas]}: comparação justa com o modelo fine-tunado')
imprimir_tabela(avaliar_pasta(PASTA_RESULTADOS, reservadas, sufixo='_reservadas'))
print('Tabelas completas: resultados/comparacao*.csv e resultados/metricas_paginas*.csv')

In [ ]:
from ocr_historico.resultados import carregar, textos_por_arquivo

PAGINA, COMBINACOES_VER = pagina, [r['combinacao'] for r in resumo[:3]]
print('GABARITO\n', PAGINA.texto_gabarito(), '\n')
for nome in COMBINACOES_VER:
    print(nome.upper(), '\n', textos_por_arquivo(carregar(PASTA_RESULTADOS / f'{nome}.json')).get(PAGINA.arquivo), '\n')